# CONUS RZSM prediction

Generate the final Fixed EF, FNO, and LSTM root-zone soil-moisture products from the prepared ASCAT and SMAP inputs. The notebook reads the two prediction-ready neural ensemble bundles and writes one compact NetCDF per model/product pair. Neural files contain only the ensemble mean and standard deviation; individual seed predictions, yearly intermediates, manifests, and version records are not produced.

In [ ]:
from config import configure_runtime

configure_runtime()

In [ ]:
from pathlib import Path

import torch

from config import base_FP, cpuserver_data, nas_FP, das_FP, george_FP
from CONUS_RZSM_Prediction import (
    resolve_prediction_batch_sizes,
    run_conus_prediction,
)

from config import figures_FP, results_FP


## Scientific and runtime settings

Both neural models use the full 2015-04-01 through 2023-12-31 context and predict at finite satellite-observation events beginning with each pixel's 32nd valid event. FNO and LSTM share the same memory controls: automatic CPU batches are 32 pixels and 4,096 inference events; CUDA uses 64/4,096 below 20 GiB, 64/8,192 from 20 to less than 40 GiB, and 128/32,768 at 40 GiB or more. Set either override below to a positive integer to bypass the automatic value. The exponential filter uses a 15-day time constant and is vectorized over each latitude chunk, so neural pixel and inference batches do not apply to EF; its first 365 elapsed days are adjustment only, so retained EF output begins on 2016-03-31.

In [ ]:
PRODUCTS = ("ASCAT", "SMAP")
MODELS = ("EF", "FNO", "LSTM")
CONUS_BOUNDS = (-126.0, -66.0, 24.0, 51.0)
EF_TIME_CONSTANT_DAYS = 15.0
EF_SPINUP_DAYS = 365

# Memory/performance controls do not change the scientific calculation.
# None selects the device-aware defaults described above.
CHUNK_LATITUDE = 10
PIXEL_BATCH_OVERRIDE = None
INFERENCE_BATCH_OVERRIDE = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIXEL_BATCH, INFERENCE_BATCH = resolve_prediction_batch_sizes(
    DEVICE,
    pixel_batch=PIXEL_BATCH_OVERRIDE,
    inference_batch=INFERENCE_BATCH_OVERRIDE,
)

## Input and output paths

`Model_input_static_eqd_010.nc` is the already prepared seven-layer static input plus `CONUS_mask`; this prediction notebook does not rebuild preprocessing products.

In [ ]:
RESULT_FP = Path(results_FP)
CONUS_RESULT_FP = RESULT_FP / "CONUS_Prediction"
MODEL_INPUT_FILE = CONUS_RESULT_FP / "Model_input_static_eqd_010.nc"

PRODUCT_FILES = {
    "ASCAT": RESULT_FP / "ASCAT" / "ASCAT_20150401_20231231_eqd_010.nc",
    "SMAP": RESULT_FP / "SMAP" / "SMAP_20150401_20231231_eqd_010.nc",
}
FNO_BUNDLE_FILES = {
    product: RESULT_FP / "FNO" / "Train" / f"FNO_{product}_ensemble.pt"
    for product in PRODUCTS
}
LSTM_BUNDLE_FILES = {
    product: RESULT_FP / "LSTM" / "Train" / f"LSTM_{product}_ensemble.pt"
    for product in PRODUCTS
}
CONUS_RESULT_FP.mkdir(parents=True, exist_ok=True)

In [ ]:
required_files = [MODEL_INPUT_FILE]
for product in PRODUCTS:
    required_files.extend(
        [PRODUCT_FILES[product], FNO_BUNDLE_FILES[product], LSTM_BUNDLE_FILES[product]]
    )
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Missing CONUS prediction inputs:\n" + "\n".join(map(str, missing_files)))
print(f"Device: {DEVICE}")
print(
    f"Prediction batches: {CHUNK_LATITUDE} latitude rows, "
    f"{PIXEL_BATCH} pixels, {INFERENCE_BATCH} inference events"
)
print(f"Output directory: {CONUS_RESULT_FP}")

## Run the six final products

Each final file covers the CONUS grid only and retains the complete daily time coordinate. Values occur only on the source satellite's valid observation dates.

In [ ]:
prediction_files = {}
for product in PRODUCTS:
    prediction_files[product] = run_conus_prediction(
        product=product,
        product_file=PRODUCT_FILES[product],
        static_file=MODEL_INPUT_FILE,
        output_directory=CONUS_RESULT_FP,
        models=MODELS,
        fno_bundle_file=FNO_BUNDLE_FILES[product],
        lstm_bundle_file=LSTM_BUNDLE_FILES[product],
        bounds=CONUS_BOUNDS,
        ef_time_constant_days=EF_TIME_CONSTANT_DAYS,
        ef_spinup_days=EF_SPINUP_DAYS,
        chunk_latitude=CHUNK_LATITUDE,
        pixel_batch=PIXEL_BATCH,
        inference_batch=INFERENCE_BATCH,
        device=DEVICE,
    )
prediction_files